# Stage 4, tier 1: the mRFP synonymous control

This notebook asks whether nucleotide encoder representations help predict expression beyond selected sequence-composition and predicted RNA-folding features. It extends Stage 1's control using monomeric red fluorescent protein (mRFP): the coding sequences vary while the amino-acid sequence they encode stays the same. This holds protein sequence constant while we examine nucleotide variation.

Each row in `mRFP_Expression.csv` pairs a coding sequence with a supplied expression score. The original study expressed variants of the mRFP gene in *E. coli* and measured red fluorescence as a proxy for protein production, also checking its relationship to protein abundance using proteomics. Fluorescence makes the reporter's production measurable; the supplied scores are not molecule counts. [Nieuwkoop et al., 2023](https://academic.oup.com/nar/article/51/5/2363/7016452)

Synonymous codons specify the same amino acid: coding-DNA triplets `GCT` and `GCC`, for example, correspond to RNA codons `GCU` and `GCC`, both encoding alanine. Such differences between rows can preserve the entire protein sequence. Protein encoders then receive identical amino-acid input, giving a negative control that cannot distinguish the variants from its sequence input. DNA and RNA encoders can receive different nucleotide sequences between variants. Converting T to U creates the RNA view of each row; it does not create the synonymous differences between rows.

We reuse Stage 1's saved, pooled embeddings and first check the constant-protein premise. We then construct named sequence controls, compare separate ridge predictors, and ask whether embeddings improve prediction after the controls have been used. The pretrained encoders stay frozen; this notebook evaluates their representations and does not train a multimodal fusion model. Each result below leads into the next comparison, ending with what the evidence supports and what remains unresolved.

**Project context:** research-plan item 5; [Gantt and Details task 18](../../docs/Research_Project_Plan_Gantt.xlsx); Overleaf Section 12.2, item 5. The [Stage 4 overview](README.md#mrfp-changing-the-nucleotide-sequence-while-keeping-the-protein-fixed) connects this control to the encoder-swap experiments.


---

## Setup

Prepare the Python environment using the root README's [notebook dependencies](../../README.md#notebook-dependencies) and [additional ViennaRNA dependency](../../README.md#additional-dependency-for-the-stage-4-mrfp-control). Select that environment as this notebook's kernel. Package installation belongs in that setup step; the cells below do not install packages.

### Imports

In [3]:
# Files and project paths
import os
import sys
from pathlib import Path

# Numerical arrays (NumPy) and result tables (pandas).
import numpy as np
import pandas as pd

# RNA folding: ViennaRNA supplies energy features calculated later as control inputs.
import RNA

# Compare feature-expression rankings and score held-out predictions.
from scipy.stats import spearmanr
from sklearn.metrics import r2_score

# Regression, feature scaling and cross-validation for the predictors.
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Table display width in characters, unrelated to nucleotide-window lengths.
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("ViennaRNA", RNA.__version__)

ViennaRNA 2.7.2


### Project paths and shared code

Locate the project's `Code` directory and import the shared helpers used to load data, read saved embeddings and evaluate representations. Later input and result paths are anchored to this directory.

In [4]:
# Search the kernel's working folder and its ancestors for either layout:
# Code/shared_code from the repository root, or shared_code from inside Code.
working_dir = Path.cwd().resolve()
for parent in (working_dir, *working_dir.parents):
    if (parent / "shared_code").is_dir():
        CODE_DIR = parent
        break
    if (parent / "Code" / "shared_code").is_dir():
        CODE_DIR = parent / "Code"
        break
else:
    # A for-loop's else runs only when no match ends the search with break.
    raise FileNotFoundError(
        "Cannot locate Code/shared_code. Start this notebook from within the repository."
    )

# Let Python find shared_code without changing the kernel's working folder.
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Reuse project helpers for data, saved embeddings, sequence features and analysis.
from shared_code import analysis, datasets, embeddings, sequences

# Encoder settings and folds that keep identical sequences together.
from shared_code.encoders import ENCODERS
from shared_code.splits import grouped_folds

### Settings and file locations

Reuse Stage 1's 1,000-row sample, selected from the supplied dataset of 1,459 rows in the **Pipeline control: synonymous recoding** section of [Stage 1's stability notebook](../stage1/Stage1_stability.ipynb). Sampling seed 42 reproduces that sample and identifies its matching saved embeddings. Fold-assignment seeds divide the **1,000 sampled rows** into five folds for evaluation. Different assignments rearrange their fold membership; they do not select another sample from the full dataset.

The settings below also retain the existing folding windows and result locations. The controls section explains the windows, and the prediction sections explain the fold assignments. Changing a calculation would require checking whether its saved result files remain applicable; the existing reuse checks do not track every setting.


In [5]:
# Reproduce the Stage 1 sample associated with the saved embedding directory.
CONTROL = "mrfp_expression"
N_ROWS = 1000
SEED = 42

# Compare Stage 1's pinned partition with assignments using seeds 0 through 9.
PROBE_SEED = 42
N_ASSIGNMENTS = 10

# Prefix lengths in nucleotides, measured from the coding sequence's start.
TRACK_A_WINDOW = 200  # Nussinov pairing fraction, inherited from Stage 1.
START_WINDOW = 60     # ViennaRNA folding energy near the start codon.

# Read Stage 1 inputs and keep this notebook's result tables in Stage 4.
EMBEDDING_DIR = (
    CODE_DIR / "stage1" / "embeddings" / "mrfp_expression_n1000_seed42"
)
RESULTS_DIR = CODE_DIR / "stage4" / "results" / "tier1_mrfp"
STRUCTURE_CACHE = RESULTS_DIR / "structure_features.csv"
PROBE_CACHE = RESULTS_DIR / "probe_scores.csv"
GAIN_CACHE = RESULTS_DIR / "gains_over_controls.csv"

### Prepare result storage

Prepare the destination and helper for the result tables written during the later calculations. Each calculation decides which previously calculated results can be reused. The saving helper writes the updated result table to its destination file.

In [6]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def save(frame, path):
    """Write a table so that an interruption leaves the previous complete copy."""
    # Finish the new CSV before replacing the previous complete table.
    partial = path.with_name(path.name + ".partial")
    frame.to_csv(partial, index=False)
    os.replace(partial, path)

### Load the sample and saved embeddings

Load the coding sequences and expression scores, then read their pooled embeddings from Stage 1. Each embedding is one numerical vector per sequence. All feature matrices must describe the same rows in the same order; the saved-input checks compare their identity with the requested sequences.

This cell reads saved representations without loading or running pretrained encoders. Missing or incompatible embeddings must be resolved with the Stage 1 producer. The output checks the sample, sequence lengths, expression range and matrix widths before any prediction is evaluated.


In [7]:
# Keep sequences, expression targets and split groups in the loader's row order.
control = datasets.load_dataset(CONTROL, n_rows=N_ROWS, seed=SEED)
seqs = control.sequences
y = control.labels.astype(float)
groups = control.groups  # Identical sequences must remain in the same fold.

# Labels describe the results; each embedding matrix supplies numerical inputs.
MODALITY = {e.key: e.modality for e in ENCODERS.values()}
LABEL = {e.key: e.label for e in ENCODERS.values()}
EMBEDDINGS = {
    key: embeddings.read_saved(encoder, seqs, EMBEDDING_DIR)
    for key, encoder in ENCODERS.items()
}

# Check the sample and cached matrix widths before constructing comparisons.
print(
    f"{len(seqs)} rows ({control.n_sampled} sampled), {len(set(seqs))} distinct sequences, "
    f"lengths {sorted({len(s) for s in seqs})}, expression {y.min():.2f} to {y.max():.2f}"
)
print(
    f"{len(EMBEDDINGS)} cached embeddings read: "
    + ", ".join(f"{k} {v.shape[1]}" for k, v in EMBEDDINGS.items())
)

1000 rows (1000 sampled), 997 distinct sequences, lengths [678], expression 7.62 to 11.38
12 cached embeddings read: nt500m 1280, ntv2_100m 512, dnabert2 768, hyenadna 256, rnafm 640, rinalmo 640, mrnafm 1280, calm 768, esm2_8m 320, esm2_35m 480, esm2_150m 640, protbert 1024


The displayed sample includes repeated nucleotide sequences, and all its coding sequences have the same length. `seqs`, `y` and `groups` keep the sequences, expression targets and evaluation groups aligned; `EMBEDDINGS` supplies the matching predictor inputs. Identical sequences will stay in the same validation fold. We can now check whether the nucleotide variants also preserve one translated protein.


---

## Check the constant-protein premise

Before predicting expression, translate the sampled nucleotide sequences and inspect how much their cached representations vary. The intended control has one amino-acid sequence shared by all variants: protein encoders receive identical input, while nucleotide encoders can receive different input.

For each encoder, the centroid is its average embedding. The table reports the mean Euclidean distance of its rows from that centroid. This describes variation within that representation; differently scaled embedding spaces cannot be ranked for predictive quality by these distances.


In [3]:
# Check that nucleotide variation preserves one translated protein sequence.
proteins = {sequences.translate_cds(s) for s in seqs}

# Measure each encoder's variation around its own average embedding.
spread = pd.DataFrame(
    {
        "modality": [MODALITY[k] for k in EMBEDDINGS],
        "width": [EMBEDDINGS[k].shape[1] for k in EMBEDDINGS],
        "mean distance to centroid": [
            analysis.centroid_spread(EMBEDDINGS[k]) for k in EMBEDDINGS
        ],
    },
    index=[LABEL[k] for k in EMBEDDINGS],
)
print(f"distinct proteins: {len(proteins)}")
spread.round(5)

distinct proteins: 1


,modality,width,mean distance to centroid
NT 500M human-ref,DNA,1280,4.72361
NT v2 100M multi-species,DNA,512,1.26535
DNABERT-2,DNA,768,0.74851
HyenaDNA large,DNA,256,0.61138
RNA-FM,RNA,640,0.81823
RiNALMo 150M,RNA,640,3.96281
mRNA-FM,RNA,1280,1.74951
CaLM,RNA,768,1.99879
ESM-2 8M,protein,320,0.00004
ESM-2 35M,protein,480,0.00004


The diagnostic reports one translated protein and very small spread in the protein embeddings, alongside variation in nucleotide embeddings. This is consistent with the intended synonymous control. Embedding spread alone does not establish cache provenance; the earlier saved-input checks provide that identity comparison.

The nucleotide representations vary, but this check does not say what that variation predicts. We next construct features with named sequence meanings so their predictive usefulness can be compared with the embeddings.


---

## Build sequence controls

The controls summarize properties of each nucleotide sequence. They give us a reference for asking whether embeddings improve expression prediction beyond these selected summaries. Amino-acid composition and sequence length are constant in this sample, so they are omitted.

RNA can form base pairs within its own sequence, creating secondary structure. Nussinov summarizes the maximum fraction of bases that can be paired under its pairing rules. ViennaRNA estimates minimum folding free energy, in kcal/mol, using a model of the energetic favourability of RNA secondary structures. This energy is calculated from the sequence, not obtained from the fluorescence measurements. Neither calculation measures these RNAs' structures inside cells.

| Control input | Meaning in this comparison |
| --- | --- |
| GC | Fraction of nucleotide positions occupied by G or C. |
| GC3 | Fraction of third codon positions occupied by G or C. |
| Codon use | Frequencies of the 64 codons, supplying one feature column per codon. |
| Nussinov, first 200 nt | Paired-base fraction in the retained Stage 1 prefix. |
| MFE, first 60 nt | ViennaRNA minimum folding free energy for the coding sequence's initial region. |
| MFE, whole sequence | The same energy calculation over the entire coding sequence. |

The 200-nucleotide Nussinov window and the 60-nucleotide ViennaRNA window both begin at the coding sequence's start. Their lengths retain the existing analysis choices; this notebook does not compare window lengths to establish an optimum.

### Calculate or reuse folding features

Folding produces the three structural summaries used alongside composition features. Calculate uncached sequences in batches, then restore sampled row order so every result aligns with the expression scores and saved embeddings. The cache matches sequences; it does not establish whether changed folding settings would reproduce the stored values.


In [ ]:
def structure_row(seq):
    # ViennaRNA uses RNA bases; convert the coding DNA view from T to U.
    rna = sequences.dna_to_rna(seq)

    # RNA.fold returns (structure, energy); [1] retains the energy in kcal/mol.
    return {
        "sequence": seq,
        "nussinov_200": sequences.nussinov_pairing_fraction(seq[:TRACK_A_WINDOW]),
        "mfe_start_60": RNA.fold(rna[:START_WINDOW])[1],
        "mfe_full": RNA.fold(rna)[1],
    }


# Existing folding rows are matched by sequence, without checking changed settings.
done = pd.read_csv(STRUCTURE_CACHE) if STRUCTURE_CACHE.exists() else pd.DataFrame()
have = set(done["sequence"]) if len(done) else set()

# Calculate each uncached sequence once, preserving first-occurrence order.
todo = [s for s in dict.fromkeys(seqs) if s not in have]
for start in range(0, len(todo), 50):
    # Save after each batch of newly calculated sequences.
    chunk = pd.DataFrame([structure_row(s) for s in todo[start:start + 50]])
    done = pd.concat([done, chunk], ignore_index=True)
    save(done, STRUCTURE_CACHE)

# Restore sampled row order, including duplicates, to align with y and embeddings.
structure = done.set_index("sequence").loc[seqs].reset_index(drop=True)
# The 'read back' count includes repeated sample rows, not just cached sequences.
print(f"structure features: {len(todo)} computed now, {len(seqs) - len(todo)} read back")


structure features: 997 computed now, 3 read back


### Define the control inputs

Combine the folding results with GC, GC3 and codon frequencies. `CONTROLS` holds the matrix for each individual predictor. The scalar-feature table, `FEATURES`, is derived from those same matrices, so a control's calculation has one definition.


In [ ]:
# Skip GC, GC3 and log length in composition_features; take its 64 codon frequencies.
codon_use = np.array([sequences.composition_features(s)[3:3 + 64] for s in seqs])

# Define each control once as a matrix: sampled rows by feature columns.
# Reshape scalar arrays, or select a one-column DataFrame, to retain two dimensions.
CONTROLS = {
    "GC": np.array([sequences.gc_content(s) for s in seqs]).reshape(-1, 1),
    "GC3": np.array([sequences.gc3_content(s) for s in seqs]).reshape(-1, 1),
    "codon use (64)": codon_use,
    "Nussinov, first 200 nt": structure[["nussinov_200"]].to_numpy(),
    "MFE, first 60 nt": structure[["mfe_start_60"]].to_numpy(),
    "MFE, whole sequence": structure[["mfe_full"]].to_numpy(),
}

# Scalar summaries use the same definitions; the 64-column codon block is excluded.
FEATURES = pd.DataFrame({
    name: values[:, 0]
    for name, values in CONTROLS.items()
    if values.shape[1] == 1
})


### Group the controls and inspect their associations

The **Track A controls** combine GC3, codon frequencies and the Nussinov fraction. The expanded **all controls** group contains the Track A features plus GC content and both folding-energy estimates. We fit one ridge predictor using the Track A features and another using the expanded group. The later incremental comparison tests whether embeddings add predictive value above each control group.

The table below gives each scalar feature's range and Spearman rank association with expression. The codon block supplies 64 predictor columns and is excluded from this one-value-per-feature summary. Association describes how measured values vary together; held-out prediction is evaluated next.


In [4]:
# Membership is explicit: adding a diagnostic control must not change these groups.
track_a_members = (
    "GC3",
    "codon use (64)",
    "Nussinov, first 200 nt",
)
control_groups = {
    "Track A controls": track_a_members,
    "all controls": (
        "GC",
        *track_a_members,
        "MFE, first 60 nt",
        "MFE, whole sequence",
    ),
}

# Join columns in the stated order, keeping every matrix aligned to the same rows.
CONTROL_SETS = {
    name: np.hstack([CONTROLS[member] for member in members])
    for name, members in control_groups.items()
}

# Describe association with expression; no prediction model is fitted here.
pd.DataFrame({
    "Spearman with expression": [
        spearmanr(FEATURES[c], y).statistic for c in FEATURES
    ],
    "min": FEATURES.min(),
    "max": FEATURES.max(),
}).round(3)


,Spearman with expression,min,max
GC,-0.173,0.445,0.546
GC3,-0.249,0.376,0.677
"Nussinov, first 200 nt",0.132,0.650,0.810
"MFE, first 60 nt",0.264,-21.800,-2.800
"MFE, whole sequence",-0.144,-229.400,-155.900


The first-60-nucleotide folding energy and GC3 have the largest absolute rank associations in this table, with opposite signs. These associations do not establish causation or show that the features supply distinct information from one another. They also do not establish that the folding predictions match cellular RNA structures.

We now have individual controls, two explicitly defined control groups and the saved embeddings. The next comparison asks how well each predicts expression on rows held out from fitting.


---

## Compare separate predictors

Fit a separate ridge predictor to each encoder's pooled embeddings, each individual control, each combined control set and a random-feature reference. Each input matrix has one row per sampled sequence; its columns are the features available to that predictor. The pretrained encoders remain frozen.

Protein encoders provide a reference without sequence input that distinguishes these synonymous variants. The fixed random-feature matrix supplies another reference, with 640 Gaussian columns. It is one width and one random draw: assignment seeds change the folds, not these features. The cell below collects these comparisons before any fitting.


In [ ]:
# Fix the random-feature reference once; assignment seeds below only change folds.
rng = np.random.default_rng(0)
NULL = rng.normal(size=(len(seqs), 640))

# Every candidate matrix follows the same row order as expression y and groups.
MODELS = {
    **{LABEL[k]: X for k, X in EMBEDDINGS.items()},
    **CONTROLS,
    **CONTROL_SETS,
    "random features (640)": NULL,
}
KIND = {
    **{LABEL[k]: MODALITY[k] for k in EMBEDDINGS},
    **{c: "control" for c in (*CONTROLS, *CONTROL_SETS)},
    "random features (640)": "null",
}
SEEDS = (PROBE_SEED, *range(N_ASSIGNMENTS))


### Evaluate on grouped folds

Each fold assignment divides the **1,000 sampled rows** into five folds, keeping identical sequences together. The predictor fits on four folds and is evaluated on the fifth. This repeats until every fold has served as the held-out fold. Feature scaling and predictor fitting use the outer training rows, and expression is scored on the held-out rows. Stage 1's pinned assignment uses seed 42; the ten further assignments use seeds 0 through 9.

The [linear-probing guide](../../docs/methods/linear-probing.md) explains scaling, ridge penalties and the current limitation of internal penalty selection. The loop reuses or calculates assignment scores for each input matrix, then saves that model's results.


In [ ]:
scores = pd.read_csv(PROBE_CACHE) if PROBE_CACHE.exists() else pd.DataFrame()
for name, X in MODELS.items():
    # Reuse is based on the saved row count, not a full input/settings identity check.
    if len(scores) and (scores["model"] == name).sum() == len(SEEDS):
        continue

    # Store one mean of five held-out fold scores for each assignment.
    rows = [
        {
            "model": name,
            "seed": s,
            "r2": analysis.probe_scores(X, y, groups=groups, seed=s).mean(),
        }
        for s in SEEDS
    ]
    # Replace this model's previous rows and save the updated comparison table.
    scores = pd.concat([
        scores[scores["model"] != name] if len(scores) else scores,
        pd.DataFrame(rows),
    ], ignore_index=True)
    save(scores, PROBE_CACHE)


### Read the individual prediction scores

The prediction score for a held-out fold is:

$$
R^2 = 1 - \frac{\sum_i (y_i-\hat y_i)^2}{\sum_i (y_i-\bar y)^2}.
$$

In this notation:

- $i$ indexes the rows in that held-out fold.
- $y_i$ is the measured expression score and $\hat y_i$ is its prediction.
- $\bar y$ is the mean measured expression within that held-out fold.
- Each sum adds squared deviations over those same rows.

A score of zero matches the squared error of predicting that fold's mean expression; a negative score has larger error than that reference. `analysis.probe_scores` returns five fold scores, and the notebook averages them for each assignment.

**Pinned** uses Stage 1's fixed assignment. The remaining score columns give the mean and standard deviation of the five-fold means over ten further assignments. That spread describes sensitivity to partitioning the same sample; it is not a confidence interval or variation across independent datasets.


In [5]:
# Separate the pinned result from variation across the further assignments.
pinned = scores[scores["seed"] == PROBE_SEED].set_index("model")["r2"]
further = scores[scores["seed"] != PROBE_SEED].groupby("model")["r2"]
PROBES = pd.DataFrame({
    "kind": pd.Series(KIND),
    "R2, pinned": pinned,
    "R2, 10 assignments": further.mean(),
    "SD over assignments": further.std(ddof=0),
}).loc[list(MODELS)]
PROBES.round(3)


,kind,"R2, pinned","R2, 10 assignments",SD over assignments
NT 500M human-ref,DNA,0.296,0.291,0.013
NT v2 100M multi-species,DNA,0.281,0.264,0.016
DNABERT-2,DNA,0.226,0.206,0.012
HyenaDNA large,DNA,0.249,0.247,0.021
RNA-FM,RNA,0.222,0.223,0.008
RiNALMo 150M,RNA,0.285,0.284,0.019
mRNA-FM,RNA,0.463,0.466,0.013
CaLM,RNA,0.298,0.304,0.021
ESM-2 8M,protein,-0.003,-0.005,0.002
ESM-2 35M,protein,-0.003,-0.005,0.002


The controls predict some expression variation: codon frequencies alone average approximately $R^2=0.184$, and the first-60-nucleotide folding energy scores $0.108$. The Track A group scores $0.190$, rising to $0.306$ when GC and both energies are added. This does not isolate which added feature accounts for the combined improvement or establish an optimal folding window.

Among the pooled representations, mRNA-FM has the highest mean over the ten further assignments, approximately $R^2=0.466$, followed by CaLM at $0.304$ and NT 500M at $0.291$. Protein predictors and the random-feature reference have slightly negative mean scores, consistent with their roles as references without useful variant-specific sequence input.

These separate scores leave an important question open: do the embeddings improve predictions beyond what the controls already capture? A strong individual score alone cannot answer that. The next comparison fits an additional predictor to errors left by the controls.


---

## Test what the embeddings add above the controls

Retain the same rows and control sets, and add an embedding-based predictor to one already using the controls. `C` contains control features, `X` contains one encoder's embeddings or the random features, and `y` contains expression scores; all three follow the same sampled row order.

**Construct the additional predictor's target.** Within each outer training subset, make control predictions through an inner set of grouped folds. For each training row, subtract the prediction made without fitting on its sequence group:

$$
r_i = y_i - \hat y_{i,\mathrm{inner}}.
$$

In this notation:

- $i$ identifies an outer-training row and $y_i$ is its measured expression.
- $\hat y_{i,\mathrm{inner}}$ is the control prediction from an inner fold that held out that row's sequence group.
- $r_i$ is the residual expression error used to train the predictor on `X`.

This gives the additional predictor a target based on held-out control errors. The outer test rows remain outside both training procedures.

**Evaluate the combined prediction.** Refit the control predictor on all outer-training rows. Train the additional predictor using encoder embeddings (or the fixed random features) as inputs and the previously calculated, held-out control errors as targets. Those residual targets come from the earlier inner-fold predictions. Use both fitted predictors to predict the outer test rows, add their predictions, and compare the combined score with the controls alone:

$$
\hat y_{j,\mathrm{combined}} = \hat y_{j,\mathrm{controls}} + \hat r_j,
\qquad
\Delta R^2 = R^2_{\mathrm{combined}} - R^2_{\mathrm{controls}}.
$$

In this notation:

- $j$ identifies an outer-test row.
- $\hat y_{j,\mathrm{controls}}$ is the control predictor's expression prediction, and $\hat r_j$ is the additional predictor's estimated residual.
- $\hat y_{j,\mathrm{combined}}$ is their sum.
- The two $R^2$ scores use the same outer-test rows; $\Delta R^2$ is their difference.

The two predictors fit their penalties separately. This evaluates a particular residual-prediction procedure, which differs from fitting a single ridge model to concatenated controls and embeddings. The explicitly grouped outer and residual-prediction folds do not change `RidgeCV`'s internal penalty-selection defaults; the [evaluation limitation](../../docs/methods/linear-probing.md#current-evaluation-limitation) still applies.


In [ ]:
def ridge():
    # Use a fresh scaler and model for each independent fit, retaining the same grid.
    return Pipeline([
        ("scale", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-3, 5, 20))),
    ])


def gain_over(C, X, seed):
    """Held-out R^2 of the controls alone and of the controls plus X, on the same folds."""
    base, stacked = [], []
    for train, test in grouped_folds(groups, seed=seed):
        # Inner indices refer to the outer-training subset, not the complete dataset.
        inner_groups = [groups[i] for i in train]
        inner = grouped_folds(inner_groups, seed=f"inner-{seed}")

        # Form training residuals from control predictions held out by sequence group.
        honest = cross_val_predict(ridge(), C[train], y[train], cv=inner)

        # Predict the same test rows with controls and a model trained on residual targets.
        controls = ridge().fit(C[train], y[train]).predict(C[test])
        extra = ridge().fit(X[train], y[train] - honest).predict(X[test])
        base.append(r2_score(y[test], controls))
        stacked.append(r2_score(y[test], controls + extra))

    # Each returned value is an unweighted mean over the five outer folds.
    return float(np.mean(base)), float(np.mean(stacked))


### Evaluate the paired comparison

Apply this procedure to every encoder and the same random-feature reference, above each of the two control groups. Both sides of each comparison use the same test rows. Across the ten assignments, only the fold partition changes; the sampled rows, feature definitions and residual-prediction method stay fixed.

The loop saves after completing the assignments for each candidate/control pair. Its existing reuse check is based on the number of saved rows for that pair.


In [ ]:
# Compare each encoder, plus the same random reference, above each control set.
CANDIDATES = {
    **{LABEL[k]: X for k, X in EMBEDDINGS.items()},
    "random features (640)": NULL,
}

gains = pd.read_csv(GAIN_CACHE) if GAIN_CACHE.exists() else pd.DataFrame()
for set_name, C in CONTROL_SETS.items():
    for name, X in CANDIDATES.items():
        # The existing reuse check counts saved rows for this candidate/control pair.
        if len(gains) and (
            (gains["model"] == name) & (gains["controls"] == set_name)
        ).sum() == N_ASSIGNMENTS:
            continue

        rows = []
        for s in range(N_ASSIGNMENTS):
            base, stacked = gain_over(C, X, s)
            rows.append({
                "model": name,
                "controls": set_name,
                "seed": s,
                "controls R2": base,
                "with encoder R2": stacked,
            })
        # Save after all requested assignments for this candidate/control pair.
        gains = pd.concat([gains, pd.DataFrame(rows)], ignore_index=True)
        save(gains, GAIN_CACHE)


### Read the added predictive value

`gain_over` averages the five outer-fold scores for each assignment. Within each fold assignment, subtract the controls-only mean $R^2$ from the combined prediction's mean $R^2$. This produces ten gains, one per assignment. The table reports their average (**gain**), their **lowest** and **highest** values, and how many are positive (**assignments above 0**). A positive gain supports predictive usefulness beyond the selected controls under this procedure; it does not identify a biological mechanism or exhaust the possible sequence controls.


In [6]:
# Subtract paired assignment means before summarizing their variation.
gains["gain"] = gains["with encoder R2"] - gains["controls R2"]
GAINS = (
    gains.groupby(["controls", "model"], sort=False)
    .agg(**{
        "controls R2": ("controls R2", "mean"),
        "with encoder R2": ("with encoder R2", "mean"),
        "gain": ("gain", "mean"),
        "lowest": ("gain", "min"),
        "highest": ("gain", "max"),
        "assignments above 0": ("gain", lambda g: int((g > 0).sum())),
    })
)
GAINS.insert(0, "kind", [KIND[m] for _, m in GAINS.index])
GAINS.round(3)


kind  controls R2  with encoder R2   gain  lowest  highest  assignments above 0
controls         model                                                                                                       
Track A controls NT 500M human-ref             DNA        0.190            0.320  0.130   0.118    0.146                   10
                 NT v2 100M multi-species      DNA        0.190            0.251  0.061   0.025    0.079                   10
                 DNABERT-2                     DNA        0.190            0.233  0.044   0.028    0.058                   10
                 HyenaDNA large                DNA        0.190            0.266  0.077   0.048    0.098                   10
                 RNA-FM                        RNA        0.190            0.244  0.054   0.041    0.065                   10
                 RiNALMo 150M                  RNA        0.190            0.287  0.098   0.068    0.120                   10
                 mRNA-FM                       RNA        0.190            0.451  0.261   0.231    0.290                   10
                 CaLM                          RNA        0.190            0.282  0.093   0.064    0.121                   10
                 ESM-2 8M                  protein        0.190            0.189 -0.000  -0.000    0.000                    3
                 ESM-2 35M                 protein        0.190            0.189 -0.000  -0.000    0.000                    3
                 ESM-2 150M                protein        0.190            0.189 -0.000  -0.000    0.000                    3
                 ProtBERT                  protein        0.190            0.189 -0.000  -0.000    0.000                    3
                 random features (640)        null        0.190            0.188 -0.001  -0.003    0.000                    1
all controls     NT 500M human-ref             DNA        0.306            0.381  0.076   0.067    0.088                   10
                 NT v2 100M multi-species      DNA        0.306            0.334  0.029   0.010    0.048                   10
                 DNABERT-2                     DNA        0.306            0.332  0.026   0.010    0.034                   10
                 HyenaDNA large                DNA        0.306            0.342  0.036   0.005    0.057                   10
                 RNA-FM                        RNA        0.306            0.327  0.021   0.011    0.030                   10
                 RiNALMo 150M                  RNA        0.306            0.355  0.049   0.033    0.067                   10
                 mRNA-FM                       RNA        0.306            0.488  0.182   0.168    0.203                   10
                 CaLM                          RNA        0.306            0.328  0.022  -0.001    0.043                    9
                 ESM-2 8M                  protein        0.306            0.306 -0.000  -0.000    0.000                    3
                 ESM-2 35M                 protein        0.306            0.306 -0.000  -0.000    0.000                    3
                 ESM-2 150M                protein        0.306            0.306 -0.000  -0.000    0.000                    3
                 ProtBERT                  protein        0.306            0.306 -0.000  -0.000    0.000                    3
                 random features (640)        null        0.306            0.305 -0.001  -0.004    0.001                    2

Every nucleotide encoder has positive gain above Track A in all ten assignments. Above all controls, each still has a positive mean gain; seven gain in every assignment, while CaLM gains in nine. Protein gains are negligible at the displayed precision, and the random reference has a slightly negative mean gain.

mRNA-FM has the largest mean gain above all controls: the combined prediction scores approximately $R^2=0.488$, compared with $0.306$ for controls alone, giving a gain of $0.182$. NT 500M follows with a gain of about $0.076$. This gain compares the residual-based combination with controls alone; it is not the difference between the individual-predictor scores in the previous table.

The result answers the opening question for this procedure: the selected controls leave expression variation that the nucleotide embeddings help predict. It does not identify a causal mechanism or establish that all composition and RNA-structure information has been accounted for.


---

## Findings, limitations and connection to Stage 4

For this sample, nucleotide representations add predictive usefulness beyond the selected sequence and folding controls under the specified ridge procedure. The controls themselves are informative, and mRNA-FM gives the strongest results in both the individual and incremental comparisons. The protein reference behaves as expected when synonymous variants share the same amino-acid input.

This evidence concerns a sample of synonymous variants of one gene, pooled embeddings and the chosen linear predictors. Repeated fold assignments repartition the same examples; they do not establish performance on independent genes or datasets. The gains do not identify a biological mechanism, validate the predicted RNA structures inside cells or rule out nonlinear relationships involving the same controls. The [internal penalty-selection limitation](../../docs/methods/linear-probing.md#current-evaluation-limitation) also remains part of the evaluation.

Within Stage 4, this provides a controlled comparison of encoder representations while protein sequence stays fixed. It motivates examining promising representations in the encoder-swap experiments. Their performance with trained heads on token-level streams, including the ordering of encoders in a fusion model, remains a separate question for those experiments.

The [mRFP run record and its prose qualification](../../docs/run-records.md#2026-10-19-stage4-mrfp-control-the-mrfp-synonymous-control-research-plan-item-5) retain the execution account and earlier interpretation. This notebook's discussion draws on those existing results.
